# Explore the ELT pipeline

Follow one batch: **download → load raw data → transform with SQL → save a report**. We use the project’s existing objects and SQL, not a second implementation.

Run from the repository or `notebooks/` directory with the project's Python 3.13 kernel. Install the optional database dependency first: `uv sync --locked --extra elt`. See the [ELT guide](../docs/elt.md) for the normal commands.

This notebook uses a temporary workspace. It copies already staged files when available and downloads missing ones from the configured public source. It does not change the project's existing warehouse or reports. Temporary results disappear when the kernel exits.

In [ ]:
from pathlib import Path
import shutil
import sys
import tempfile

import duckdb
import pandas as pd

root = Path.cwd()
if not (root / 'pyproject.toml').is_file():
    root = root.parent
if not (root / 'pyproject.toml').is_file():
    raise FileNotFoundError('Open this notebook from the repository or notebooks folder.')
sys.path.insert(0, str(root / 'src'))

## 1. Choose a period

Change the month and duration to explore another period. Rerun this cell and every following cell after changing them. One month keeps the example quick.

In [ ]:
from green_taxi.settings import PipelineSettings
from green_taxi.elt.pipeline import ELTPipeline

workspace = tempfile.TemporaryDirectory(prefix='green-taxi-elt-')
settings = PipelineSettings(
    data_dir=Path(workspace.name),
    reporting_period={'start_month': '2025-01', 'duration_months': 1},
)
pipeline = ELTPipeline(settings)
print(f'Months: {", ".join(settings.months)}')
print(f'Pickup dates: {settings.reporting_period.start} to {settings.reporting_period.end} (end excluded)')
print(f'Temporary workspace: {workspace.name}')

## 2. Stage the files

Copy available files to the temporary workspace. The existing downloader checks them and downloads any missing month. Refreshing this notebook never overwrites the original files.

In [ ]:
settings.raw_dir.mkdir(parents=True, exist_ok=True)
for month in settings.months:
    filename = f'green_tripdata_{month}.parquet'
    source = root / 'data' / 'raw' / filename
    if source.is_file():
        shutil.copy2(source, settings.raw_dir / filename)

context = pipeline.start_run()
inputs = pipeline.downloader.download()
for item in inputs:
    print(f'{item.path.name}: {"downloaded" if item.downloaded else "reused copy"}')

## 3. Load raw rides

The warehouse stores each file's original rows before applying the revenue rules. The preview below shows a few rides from the first selected month; it is not the final report.

In [ ]:
pipeline.load(inputs)
table = f'source_{settings.months[0].replace("-", "_")}'
with duckdb.connect(str(pipeline.database_path), read_only=True) as connection:
    raw_count = connection.execute(f'SELECT count(*) FROM raw."{table}"').fetchone()[0]
    preview = connection.execute(
        f'SELECT lpep_pickup_datetime, total_amount FROM raw."{table}" LIMIT 5'
    ).df()
print(f'Raw rides in {settings.months[0]}: {raw_count:,}')
preview

## 4. Transform with SQL

The existing SQL checks dates and amounts, then groups accepted rides by pickup day. Zero and negative amounts remain valid; invalid or out-of-period rides are counted. The database and transformation rules are explained in the [ELT guide](../docs/elt.md#sql-flow).

In [ ]:
result = pipeline.calculate(inputs)
print('Ride counts:')
print(pd.Series(result.counts).to_string())
print(f'\nTotal recorded charges: ${result.total_amount:,.2f}')
result.facts.head()

## 5. Save and inspect the report

The shared writer checks and saves the daily facts, date dimension and run metadata in the temporary workspace. The metadata describes this run, not a previous run.

In [ ]:
import json

metadata_path = pipeline.save(inputs, result, context)
metadata = json.loads(metadata_path.read_text())
print(f'Report folder: {metadata_path.parent}')
print(f'Mode: {metadata["execution_mode"]}')
print(f'Rows read: {metadata["counts"]["rows_read"]:,}')
print(f'Rows accepted: {metadata["counts"]["rows_used"]:,}')
print('Files:')
for path in sorted(metadata_path.parent.iterdir()):
    print(f'  {path.name}')

## Takeaway

ELT stores the raw rides first, then uses SQL to prepare the daily report. Python coordinates the work; DuckDB executes the SQL. For the regular persistent run, use the [ELT guide](../docs/elt.md).